# Sandbox: Step-by-Step Production Pipeline

Educational Goal:
- Understand how the production pipeline works
- Execute each stage interactively
- Debug modules without editing src/

This notebook mirrors src/main.py intentionally.
Do NOT rewrite logic here.
Only orchestrate existing modules.

In [30]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

from src.utils import setup_logging
import src.load_data as load_data
import src.clean_data as clean_data
import src.validate as validate
import src.features as features
import src.train as train
import src.evaluate as evaluate
import src.infer as infer
import yaml

from src.utils import save_csv, save_json, save_model


In [31]:
def load_config(config_path: str):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)
    
SETTINGS = load_config("config.yaml")

setup_logging(
    level=SETTINGS.get("logging", {}).get("level", "INFO"),
    log_file=SETTINGS.get("logging", {}).get("file"),
)

In [32]:
print("Using SETTINGS:", SETTINGS)

Using SETTINGS: {'project': {'name': 'Spotify Sound Archetypes', 'problem_type': 'regression', 'target_column': 'popularity'}, 'paths': {'raw_data': 'data/raw/SpotifyAudioFeaturesApril2019.csv', 'clean_data': 'data/processed/clean.csv', 'model_path': 'models/model.joblib', 'report_path': 'reports/predictions.csv'}, 'train': {'test_size': 0.2, 'seed': 42, 'rf_n_estimators': 100, 'rf_max_depth': 10}, 'features': {'quantile_bin': ['duration_ms', 'tempo'], 'categorical_onehot': ['key', 'mode'], 'numeric_passthrough': ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'valence'], 'n_bins': 5}}


Infrastructure Setup

In [33]:
for folder in ["data/raw", "data/processed", "models", "reports"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Infrastructure folders ensured.")

Infrastructure folders ensured.


Load Data

In [ ]:
raw_path = Path("data/raw/SpotifyAudioFeaturesApril2019.csv")

df_raw = load_data.load_raw_data(raw_path)

print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (130663, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


Clean Data

In [35]:
df_clean = clean_data.clean_dataframe(df_raw, SETTINGS["project"]["target_column"])

save_csv(df_clean, Path(SETTINGS["paths"]["clean_data"]))

print("Clean shape:", df_clean.shape)
df_clean.head()

Clean shape: (129858, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


Validate

In [36]:
required_cols = (
        SETTINGS["features"]["quantile_bin"] + 
        SETTINGS["features"]["categorical_onehot"] + 
        SETTINGS["features"]["numeric_passthrough"] + 
        [SETTINGS["project"]["target_column"]]
    )
validate.validate_dataframe(
        df_clean,
        required_cols,
        target_column=SETTINGS["project"]["target_column"],
        allow_feature_nulls=True
    )
print("Validation passed.")

Validation passed.


Train/Test Split

In [37]:
target = SETTINGS["project"]["target_column"]
X = df_clean.drop(columns=[target])
y = df_clean[target]
    
# Accessing test_size and seed from the ['train'] block
strat = y if SETTINGS["project"]["problem_type"] == "classification" else None
X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=SETTINGS["train"]["test_size"], 
        random_state=SETTINGS["train"]["seed"], 
        stratify=strat
    )

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (103886, 16)
Test size: (25972, 16)


Feature Processor

In [38]:
preprocessor = features.get_feature_preprocessor(
        quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
        categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
        numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
        n_bins=SETTINGS["features"]["n_bins"]
    )

print("Feature recipe built.")

Feature recipe built.


Train Model

In [39]:
model_pipeline = train.train_model(
        X_train,
        y_train,
        preprocessor,
        SETTINGS["project"]["problem_type"],
        train_config=SETTINGS.get("train", {}),
    )
save_model(model_pipeline, Path(SETTINGS["paths"]["model_path"]))

print("Model trained and saved.")

Model trained and saved.


Evaluate

In [40]:
metrics = evaluate.evaluate_model(
        model_pipeline, X_test, y_test, SETTINGS["project"]["problem_type"]
    )
save_json(metrics, Path("reports/metrics.json"))
save_json(SETTINGS, Path("reports/run_config.json"))

print("Evaluation metrics:", metrics)

Evaluation metrics: {'rmse': 18.14735662010368, 'mae': 14.821598644982588}


Inference

In [41]:
df_preds = infer.run_inference(model_pipeline, X_test)
save_csv(df_preds, Path(SETTINGS["paths"]["report_path"]))

df_preds.head()

,prediction
78369,36
17504,26
59259,25
121556,17
103028,27
